In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pybedtools import BedTool

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import pysam

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]
ips_cluster_donor = list(np.loadtxt(f'{projdir}/txt/allc_table/ips_cluster_donor.txt', dtype=str))

In [ ]:
indir = f'{projdir}/pmds/dnmtools/rna_cluster_donor'
pmds = pd.read_csv(f'{indir}/merged/start_merged.filt_mcg.sorted.bed.gz', sep='\t', header=None)
pmds.index = [f'{x}_{y}' for x,y in zip(pmds[0], pmds[1])]
pmds.shape

In [ ]:
chrom_path = '/u/project/cluo/terencew/reference/hg38_igvf/GRCh38.autosomal.chromsizes'
chrom_sizes = pd.read_csv(chrom_path, sep='\t', header=None)
chrom_dict = dict(zip(chrom_sizes[0], chrom_sizes[1]))

In [ ]:
s = 'IPS_main_C29'
bins_path = f'{projdir}/bed/bins/ips_cluster_donor/{s}.chrom200_CGN-Both.full.bed.gz'
tabix = pysam.TabixFile(bins_path)

### average across all regions

In [ ]:
w_size = 1e5
tmp_sizes = [chrom_dict.get(x) for x in pmds[0]]
left_mask = pmds[1] - w_size > 0
right_mask = ((pmds[2] + w_size) - tmp_sizes) < 0
mask = left_mask & right_mask
final_pmds = pmds[mask]
final_pmds.shape

In [ ]:
n_pmds = final_pmds.shape[0]
dummy_samples = [s] * n_pmds

def process_pmds_left(chrom, pmd_start, s, b_size=200, w_size=1e5):
    ###
    bins_path = f'{projdir}/bed/bins/ips_cluster_donor/{s}.chrom{b_size}_CGN-Both.full.bed.gz'
    tabix = pysam.TabixFile(bins_path)
    
    pmd_start_bin = round(pmd_start / b_size) * b_size
    start = pmd_start_bin - w_size
    end = pmd_start_bin
    if start < 0:
        start = 0
    
    tmp_mc = [int(x.split('\t')[4]) for x in tabix.fetch(chrom, start, end)]
    tmp_cov = [int(x.split('\t')[5]) for x in tabix.fetch(chrom, start, end)]
    return tmp_mc, tmp_cov

with ProcessPoolExecutor() as executor:
    results_left = list(tqdm(executor.map(process_pmds_left, final_pmds[0], final_pmds[1], dummy_samples),
                        total=n_pmds))

In [ ]:
n_pmds = final_pmds.shape[0]
dummy_samples = [s] * n_pmds

def process_pmds_right(chrom, pmd_end, s, b_size=200, w_size=1e5):
    ###
    bins_path = f'{projdir}/bed/bins/ips_cluster_donor/{s}.chrom{b_size}_CGN-Both.full.bed.gz'
    tabix = pysam.TabixFile(bins_path)
    
    pmd_end_bin = round(pmd_end / b_size) * b_size
    start = pmd_end_bin
    end = pmd_end_bin + w_size
    
    chrom_end = chrom_dict.get(chrom)
    if end > chrom_end:
        end = chrom_end
    
    tmp_mc = [int(x.split('\t')[4]) for x in tabix.fetch(chrom, start, end)]
    tmp_cov = [int(x.split('\t')[5]) for x in tabix.fetch(chrom, start, end)]
    return tmp_mc, tmp_cov

with ProcessPoolExecutor() as executor:
    results_right = list(tqdm(executor.map(process_pmds_right, final_pmds[0], final_pmds[2], dummy_samples),
                        total=n_pmds))

In [ ]:
sum_mc = np.vstack([x[0] for x in results_left]).sum(axis=0)
sum_cov = np.vstack([x[1] for x in results_left]).sum(axis=0)
mcg = sum_mc / sum_cov
left_plot = pd.DataFrame(mcg)
left_plot['tmp'] = left_plot.index
left_plot.shape

In [ ]:
sum_mc = np.vstack([x[0] for x in results_right]).sum(axis=0)
sum_cov = np.vstack([x[1] for x in results_right]).sum(axis=0)
mcg = sum_mc / sum_cov
right_plot = pd.DataFrame(mcg)
right_plot['tmp'] = right_plot.index
right_plot.shape

### now let's load in the windows

In [ ]:
def load_pmds_windows(s):
    allc = pd.read_csv(f'{projdir}/bed/pmds/ips_cluster_donor/{s}.pmds_windows_CGN-Both.full.bed.gz', sep='\t', header=None)
    allc.index = [f'{x}_{y}' for x,y in zip(allc[0], allc[1])]
    return allc

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(load_pmds_windows, ips_cluster_donor), total=len(ips_cluster_donor)))

In [ ]:
n_windows = 500

tmp_allc = results[0]
indices = np.where(tmp_allc.index.isin(final_pmds.index) > 0)[0]

tmp_mc = np.sum([tmp_allc.iloc[idx:idx+n_windows,:][3].values for idx in indices], axis=0) 
tmp_cov = np.sum([tmp_allc.iloc[idx:idx+n_windows,:][4].values for idx in indices], axis=0)
mcg = tmp_mc / tmp_cov
pmd_plot = pd.DataFrame(mcg)
pmd_plot['tmp'] = pmd_plot.index
pmd_plot.shape

In [ ]:
left_plot.shape, right_plot.shape, pmd_plot.shape

In [ ]:
tmp_slack = 200

start = left_plot.shape[0] - tmp_slack
end = left_plot.shape[0]
tmp_left_plot = left_plot.iloc[start:end,:]

In [ ]:
start = 0
end = tmp_slack
tmp_right_plot = right_plot.iloc[start:end,:]

In [ ]:
tmp_left_plot.shape, tmp_right_plot.shape

In [ ]:
final_plot = pd.concat([left_plot, pmd_plot, right_plot])
final_plot['tmp'] = np.arange(final_plot.shape[0])
final_plot.head()

In [ ]:
### need to summarize across all regions
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'tmp'
y_lab = 0
hue = 'tmp'

ax = sns.lineplot(data=final_plot, x=x_lab, y=y_lab)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('', fontsize=16)
ax.set_title('', fontsize=20)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)

In [ ]:
final_plot = pd.concat([tmp_left_plot, pmd_plot, tmp_right_plot])
final_plot['tmp'] = np.arange(final_plot.shape[0])

### need to summarize across all regions
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'tmp'
y_lab = 0
hue = 'tmp'

ax = sns.lineplot(data=final_plot, x=x_lab, y=y_lab)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('', fontsize=16)
ax.set_title('', fontsize=20)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)

### plot across donors

In [ ]:
%%time

n_windows = 500
tmp_slack = 200

pmd_meta_results = []
for i,s in enumerate(ips_cluster_donor):
    print(s)

    ###
    dummy_samples = [s] * n_pmds
    with ProcessPoolExecutor() as executor:
        results_left = list(tqdm(executor.map(process_pmds_left, final_pmds[0], final_pmds[1], dummy_samples),
                        total=n_pmds))
    
    with ProcessPoolExecutor() as executor:
        results_right = list(tqdm(executor.map(process_pmds_right, final_pmds[0], final_pmds[2], dummy_samples),
                        total=n_pmds))
    
    ###
    sum_mc = np.vstack([x[0] for x in results_left]).sum(axis=0)
    sum_cov = np.vstack([x[1] for x in results_left]).sum(axis=0)
    mcg = sum_mc / sum_cov
    left_plot = pd.DataFrame(mcg)
    left_plot['tmp'] = left_plot.index    

    sum_mc = np.vstack([x[0] for x in results_right]).sum(axis=0)
    sum_cov = np.vstack([x[1] for x in results_right]).sum(axis=0)
    mcg = sum_mc / sum_cov
    right_plot = pd.DataFrame(mcg)
    right_plot['tmp'] = right_plot.index
    
    ###

    tmp_allc = results[i]
    indices = np.where(tmp_allc.index.isin(final_pmds.index) > 0)[0]
    tmp_mc = np.sum([tmp_allc.iloc[idx:idx+n_windows,:][3].values for idx in indices], axis=0) 
    tmp_cov = np.sum([tmp_allc.iloc[idx:idx+n_windows,:][4].values for idx in indices], axis=0)
    mcg = tmp_mc / tmp_cov
    pmd_plot = pd.DataFrame(mcg)
    pmd_plot['tmp'] = pmd_plot.index

    ###
    start = left_plot.shape[0] - tmp_slack
    end = left_plot.shape[0]
    tmp_left_plot = left_plot.iloc[start:end,:]

    start = 0
    end = tmp_slack
    tmp_right_plot = right_plot.iloc[start:end,:]
    
    ###
    final_plot = pd.concat([tmp_left_plot, pmd_plot, tmp_right_plot])
    final_plot['tmp'] = np.arange(final_plot.shape[0])
    final_plot['sample'] = s
    pmd_meta_results.append(final_plot)

In [ ]:
merged =  pd.concat(pmd_meta_results)
merged['cluster'] = ['_'.join(x.split('_')[:2]) for x in merged['sample']]
merged['donor'] = [x.split('_')[2] for x in merged['sample']]
merged.head()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
merged.to_csv(f'{outdir}/ips_cluster_donor_pmds_metagene.csv', sep='\t')

In [ ]:
!date